# MongoDB Silver — Transformation

## Transformation Decision Report

### Purpose

This notebook transforms the MongoDB Bronze video metadata dataset into the Silver layer.

The transformation focuses on:

- Flattening analytical nested structures.
- Preserving the original video grain.
- Preserving valid source values.
- Preserving intentional dirty or imperfect data.
- Adding data-quality flags instead of deleting or silently correcting records.
- Creating useful analytical date columns.

### Decisions

| Area | Decision |
|---|---|
| `_id` | Excluded because it is a MongoDB technical identifier |
| `video_id` | Preserved as the analytical video identifier |
| Nested structures | Flattened into analytical columns |
| Arrays | Preserved without exploding |
| `duration_sec` nulls | Preserved |
| Invalid duration values | Preserved and flagged |
| Missing title | Preserved and flagged |
| Live video without `match_id` | Preserved and flagged |
| Likes greater than views | Preserved and flagged |
| Invalid timestamps | Preserved and flagged if detected |
| `created_at` / `last_updated` | Preserved as timestamps with date derivatives |
| Competition teams | Preserved as an array |
| Languages and tags | Preserved as arrays |
| MongoDB internal `_id` | Not carried into Silver |

### Important Principle

Silver does not delete, impute, or silently repair intentionally imperfect records.

Questionable records remain available with explicit data-quality flags.

In [1]:
if 'spark' in globals():
    spark.stop()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/06 14:06:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/06 14:06:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-06-Sep"
TABLE_NAME = "videos"

MONGO_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

In [4]:
df = spark.read.parquet(MONGO_BRONZE_PATH)

26/09/06 14:06:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [6]:
df.printSchema()

root
 |-- _id: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- teams: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- content_type: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- last_updated: timestamp (nullable = true)
 |-- match_id: string (nullable = true)
 |-- stats: struct (nullable = true)
 |    |-- views: integer (nullable = true)
 |    |-- likes: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical: struct (nullable = true)
 |    |-- max_resolution: string (nullable = true)
 |    |-- codec: string (nullable = true)
 |-- title: s

In [5]:
df_mongo_silver = (
    df
    .select(

        # ----------------------------------------------------
        # Identifiers
        # ----------------------------------------------------

        F.col("video_id"),
        F.col("match_id"),

        # ----------------------------------------------------
        # Content
        # ----------------------------------------------------

        F.col("content_type"),
        F.col("status"),
        F.col("title"),
        F.col("description"),
        F.col("duration_sec"),

        # ----------------------------------------------------
        # Timestamps
        # ----------------------------------------------------

        F.col("created_at"),
        F.col("last_updated"),

        # ----------------------------------------------------
        # Competition
        # ----------------------------------------------------

        F.col("competition.id").alias(
            "competition_id"
        ),

        F.col("competition.name").alias(
            "competition_name"
        ),

        F.col("competition.teams").alias(
            "competition_teams"
        ),

        # ----------------------------------------------------
        # Technical
        # ----------------------------------------------------

        F.col("technical.max_resolution").alias(
            "technical_max_resolution"
        ),

        F.col("technical.codec").alias(
            "technical_codec"
        ),

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        F.col("stats.views").alias(
            "stats_views"
        ),

        F.col("stats.likes").alias(
            "stats_likes"
        ),

        # ----------------------------------------------------
        # Arrays
        # ----------------------------------------------------

        F.col("languages"),
        F.col("tags"),
    )

    # --------------------------------------------------------
    # Data-quality flags
    # --------------------------------------------------------

    .withColumn(
        "is_missing_title",
        F.col("title").isNull()
        | (F.trim(F.col("title")) == "")
    )

    .withColumn(
        "is_missing_duration",
        F.col("duration_sec").isNull()
    )

    .withColumn(
        "is_invalid_duration",
        F.col("duration_sec").isNotNull()
        & (
            (F.col("duration_sec") <= 0)
            | (F.col("duration_sec") > 10800)
        )
    )

    .withColumn(
        "is_live_without_match_id",
        (F.col("content_type") == "live")
        & F.col("match_id").isNull()
    )

    .withColumn(
        "is_likes_greater_than_views",
        F.col("stats_likes")
        > F.col("stats_views")
    )

    .withColumn(
        "is_last_updated_before_created",
        F.col("last_updated")
        < F.col("created_at")
    )

    # --------------------------------------------------------
    # Analytical date columns
    # --------------------------------------------------------

    .withColumn(
        "created_date",
        F.to_date("created_at")
    )

    .withColumn(
        "last_updated_date",
        F.to_date("last_updated")
    )
)

In [10]:
print("=" * 70)
print("MONGODB SILVER")
print("=" * 70)

df_mongo_silver.printSchema()

df_mongo_silver.show(
    10,
    truncate=False,
    vertical=True,
)

MONGODB SILVER
root
 |-- video_id: string (nullable = true)
 |-- match_id: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- competition_id: string (nullable = true)
 |-- competition_name: string (nullable = true)
 |-- competition_teams: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- technical_max_resolution: string (nullable = true)
 |-- technical_codec: string (nullable = true)
 |-- stats_views: integer (nullable = true)
 |-- stats_likes: integer (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- is_missing_title: boolean (nullable = true)


26/09/02 18:44:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 video_id                       | 01a05ead-9bba-7c46-9da6-0cf37b66a413                                                                                                                
 match_id                       | match_1427                                                                                                                                          
 content_type                   | highlight                                                                                                                                           
 status                         | ended                                                                                                                                               
 title                          | NULL                                               

Transformation Summary

In [5]:
print("=" * 70)
print("MONGODB SILVER TRANSFORMATION SUMMARY")
print("=" * 70)

print(
    "Source rows:",
    df.count()
)

print(
    "Silver rows:",
    df_mongo_silver.count()
)

print(
    "MongoDB _id removed:",
    "_id" not in df_mongo_silver.columns
)

print(
    "Nested competition flattened:",
    all(
        column in df_mongo_silver.columns
        for column in [
            "competition_id",
            "competition_name",
            "competition_teams",
        ]
    )
)

print(
    "Technical fields flattened:",
    all(
        column in df_mongo_silver.columns
        for column in [
            "technical_max_resolution",
            "technical_codec",
        ]
    )
)

print(
    "Stats fields flattened:",
    all(
        column in df_mongo_silver.columns
        for column in [
            "stats_views",
            "stats_likes",
        ]
    )
)

print(
    "Arrays preserved:",
    all(
        column in df_mongo_silver.columns
        for column in [
            "languages",
            "tags",
            "competition_teams",
        ]
    )
)

print("=" * 70)

MONGODB SILVER TRANSFORMATION SUMMARY


Source rows: 10000
Silver rows: 10000
MongoDB _id removed: True
Nested competition flattened: True
Technical fields flattened: True
Stats fields flattened: True
Arrays preserved: True


# Final Transformation Report

The MongoDB Bronze dataset was transformed into an analytical Silver representation while preserving the original video grain.

The MongoDB internal `_id` was removed because it is a source-system technical identifier and is not part of the analytical video model.

Nested competition, technical, and statistics structures were flattened into analytical columns.

Languages, tags, and competition teams were preserved as arrays because they remain meaningful multi-valued attributes.

No records were deleted.

No missing values were imputed.

No questionable values were silently corrected.

Known data-quality conditions were preserved and represented through explicit flags, including:

- missing titles
- missing durations
- invalid duration values
- live videos without match IDs
- likes greater than views
- timestamp ordering issues

The resulting dataset provides a cleaner analytical schema while preserving the original source information and intentional imperfect data for later analysis.

In [6]:
df_mongo_silver.write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-silver/mongo/videos_silver.parquet")

26/09/06 14:11:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                